# Tarea 2 · Parte 1 — Extracción: de documentos y audio a texto

**Pregunta guía:** ¿se pueden convertir documentos escaneados y entrevistas en audio en un archivo de texto estructurado y buscable?

Este notebook toma el corpus ya recolectado en `documentos/` y `audios/` y produce dos tablas:

| Archivo | Una fila por | Columnas |
|---|---|---|
| `datos/documentos.csv` | documento (cada página del PDF cuenta como uno) | `id, archivo, tipo, fuente, texto_ocr, modelo_ocr, texto_referencia` |
| `datos/transcripciones.csv` | clip de audio | `id, archivo_audio, duracion_seg, idioma, texto_asr, modelo_asr, n_hablantes, texto_referencia` |

Todo corre en local y en CPU, con modelos abiertos. Las rutas son relativas a `tarea_2/`: el notebook se ejecuta desde esa carpeta.

## 1. Contexto de la ejecución

Fecha, versiones de las librerías que hacen el trabajo pesado y un resumen de fuentes y licencias leído de `datos/fuentes.csv`.
El módulo `voz.py` de la sesión 13 se importa desde el repositorio del curso (no se copia).

In [ ]:
import hashlib
import logging
import os
import re
import sys
import time
import warnings
from datetime import datetime
from importlib.metadata import version
from pathlib import Path

os.environ["PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK"] = "True"
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.io as pio
import transformers
from huggingface_hub.utils import disable_progress_bars

sys.path.insert(0, r"C:\Users\Renat\Documentos\GitHub\Data-Science-Python\sessions\13-audio-a-texto")
import voz
from voz import SR, Transcriptor, a_srt, a_tabla, a_texto, cargar_audio, diarizar, seudonimizar, tramos_de_voz

transformers.logging.set_verbosity_error()
transformers.logging.disable_progress_bar()
disable_progress_bars()
for nombre in ("paddlex", "paddleocr", "faster_whisper"):
    logging.getLogger(nombre).setLevel(logging.ERROR)
pd.set_option("display.max_colwidth", 90)

DOCS, AUDIOS, DATOS = Path("documentos"), Path("audios"), Path("datos")

print(f"Ejecutado el {datetime.now():%Y-%m-%d %H:%M} · Python {sys.version.split()[0]} · dispositivo: {voz.dispositivo()}")
pd.Series({p: version(p) for p in ["paddleocr", "transformers", "faster-whisper", "torch"]}, name="versión").to_frame()

In [ ]:
fuentes = pd.read_csv(DATOS / "fuentes.csv")
fuentes["familia"] = fuentes.licencia.str.split(" — ").str[0].str.replace(r" del Estado.*", " (MINJUS)", regex=True)
print(f"{len(fuentes)} archivos fuente: {fuentes.modalidad.value_counts().to_dict()}")
fuentes.pivot_table(index="familia", columns="modalidad", values="id", aggfunc="count", fill_value=0, margins=True, margins_name="total")

Todo el material es propio, público o con licencia Creative Commons. Las licencias CC BY-SA obligan a atribuir y a compartir
las obras derivadas (estas transcripciones) con la misma licencia; la atribución completa por archivo está en `fuentes.csv`
y en `audios/entrevistas/atribucion.csv`.

## 2. OCR de los documentos con PaddleOCR

**Rasterizado.** El PDF de la Resolución Directoral N.º 4378-2025-JUS/DGTAIPD-DPDP trae 8 páginas. Cada una se convierte en
un PNG a 200 dpi con PyMuPDF y cuenta como un documento impreso. Los PNG quedan junto al PDF, en `documentos/impresos/`,
que Git ignora.

In [ ]:
import fitz

PDF = DOCS / "impresos" / "imp_01_resolucion_minjus_4378_2025.pdf"
with fitz.open(PDF) as pdf:
    paginas = [PDF.with_name(f"{PDF.stem}_p{i:02d}.png") for i in range(1, len(pdf) + 1)]
    for pagina, png in zip(pdf, paginas):
        pix = pagina.get_pixmap(dpi=200)
        pix.save(png)
print(f"{len(paginas)} páginas rasterizadas a {pix.width}×{pix.height} px: {paginas[0].name} … {paginas[-1].name}")

In [ ]:
TIPOS = {"impresos": "impreso", "fotos": "foto", "manuscritos": "manuscrito"}

rutas = paginas + [r for carpeta in TIPOS for r in sorted((DOCS / carpeta).glob("*.jpg"))]
docs = pd.DataFrame({"archivo": rutas})
docs["id"] = docs.archivo.map(lambda r: r.stem)
docs["tipo"] = docs.archivo.map(lambda r: TIPOS[r.parent.name])
docs["fuente_id"] = docs.id.str.replace(r"_p\d{2}$", "", regex=True)
print(f"{len(docs)} documentos: {len(paginas)} páginas del PDF + {len(rutas) - len(paginas)} imágenes")
docs.tipo.value_counts().rename("documentos").to_frame()

**Modelo.** `PaddleOCR(engine="transformers", ...)` carga la tubería clásica de PaddleOCR 3.x (detector de líneas
PP-OCRv6 + reconocedor PP-OCRv6) y la corre con PyTorch en lugar de PaddlePaddle. Las tres etapas opcionales van apagadas:

| Parámetro | Valor | Qué haría si estuviera encendido | Por qué va apagado |
|---|---|---|---|
| `use_doc_orientation_classify` | `False` | Clasifica la página en 0°, 90°, 180° o 270° y la gira | Las imágenes ya están derechas; un giro equivocado arruina la página entera |
| `use_doc_unwarping` | `False` | "Plancha" hojas curvadas o dobladas con una red de rectificado | Deforma fotos que no son hojas (latas, afiches, pizarras) y suma segundos por imagen |
| `use_textline_orientation` | `False` | Decide si cada línea está al revés y la gira 180° | Sin líneas invertidas en el corpus; solo agrega errores y tiempo |

Todas las imágenes tienen orientación EXIF 1 o ninguna, así que tampoco hay giros implícitos al leerlas.

**Un detalle de rendimiento.** En esta laptop (Intel Core i7-1355U, solo CPU) el detector tardaba 40 s por imagen y casi
todo el tiempo se iba en las convoluciones. La causa: millones de pesos del detector son números *subnormales*
(más pequeños que el menor `float32` normal), y en CPUs Intel cada operación con ellos es decenas de veces más lenta.
`torch.set_flush_denormal(True)` los trata como cero: el texto y las confianzas salen iguales y el OCR va unas 10 veces más rápido.

In [ ]:
import torch
from paddleocr import PaddleOCR

torch.set_flush_denormal(True)
ocr = PaddleOCR(engine="transformers", use_doc_orientation_classify=False, use_doc_unwarping=False,
                use_textline_orientation=False)

tuberia = ocr.paddlex_pipeline._pipeline
det, rec = tuberia.text_det_model, tuberia.text_rec_model
pesos = list(det.infer.parameters())
subnormales = sum(((p.abs() < torch.finfo(p.dtype).tiny) & (p != 0)).sum().item() for p in pesos)
print(f"Pesos subnormales en el detector: {subnormales:,} de {sum(p.numel() for p in pesos):,}")

MODELO_OCR = f"PaddleOCR {version('paddleocr')} · {det.model_name} + {rec.model_name} (engine=transformers, CPU)"
MODELO_OCR

**Lectura.** Por documento se guarda el texto concatenado de todas las líneas en orden de lectura, la confianza mediana
del reconocedor y el número de líneas. Los textos se muestran recién en la sección 4, ya seudonimizados.

In [ ]:
def leer(ruta):
    r = ocr.predict(str(ruta))[0]
    puntajes = np.asarray(r["rec_scores"], dtype=float)
    return " ".join(r["rec_texts"]), (np.median(puntajes) if puntajes.size else np.nan), len(r["rec_texts"])


t0 = time.perf_counter()
docs["texto_ocr"], docs["confianza"], docs["lineas"] = zip(*[leer(r) for r in docs.archivo])
print(f"{len(docs)} documentos leídos en {(time.perf_counter() - t0) / 60:.1f} min")
docs.groupby("tipo", sort=False).agg(documentos=("id", "size"), lineas_mediana=("lineas", "median"),
                                     confianza_mediana=("confianza", "median"),
                                     confianza_minima=("confianza", "min")).round(3)

## 3. ASR de los audios con faster-whisper

**Modelo.** `Transcriptor("faster-whisper")` de `voz.py`: los pesos de Whisper large-v3-turbo convertidos a CTranslate2 y
cuantizados a int8, que corren rápido en CPU.

**VAD siempre.** Sin detector de voz, Whisper "rellena" los silencios con frases inventadas. `Transcriptor.segmentos()` llama a
`transcribe(..., vad_filter=True)`: el VAD Silero corta el audio en tramos con voz y solo esos llegan al modelo. Por eso todos los
clips pasan por `segmentos()`. `Transcriptor.transcribir()` no activa el VAD y no se usa en ningún momento.
El mismo `segmentos()` es el que corta los 489 s de `aud_01_reunion_laboral.mp4` en tramos antes de transcribir (el audio
del video se decodifica con PyAV dentro de `cargar_audio`).

**Idioma.** Se le indica a Whisper el idioma conocido de cada clip (`es` o `en`). Whisper no conoce el quechua: en los dos
clips en quechua de Huaylas (`qwh`) se le deja detectar el idioma, sabiendo que su salida será una aproximación fonética.

**Diarización.** `diarizar()` asigna hablante (H1, H2…) a cada segmento con huellas de voz WeSpeaker y clustering. Se aplica a los
tres audios multihablante: con 2 hablantes fijos en `aud_01_reunion_laboral` y `entrevista_simulada` (se sabe cuántos son) y con el
umbral por defecto en `agustino_2`, donde el número de voces no se conoce de antemano.

In [ ]:
ATRIBUCION = pd.read_csv(AUDIOS / "entrevistas" / "atribucion.csv").set_index("archivo")
ORIGEN = {"audios": "grabación propia", "entrevistas": "Wikimedia Commons", "es_peru": "OpenSLR 73", "en_fleurs": "FLEURS"}
MULTIHABLANTE = {"aud_01_reunion_laboral": 2, "entrevista_simulada": 2, "agustino_2": None}

rutas = [AUDIOS / "aud_01_reunion_laboral.mp4", *[r for c in ["entrevistas", "es_peru", "en_fleurs"]
                                                  for r in sorted((AUDIOS / c).iterdir()) if r.suffix in {".ogg", ".wav"}]]
clips = pd.DataFrame({"archivo_audio": rutas})
clips["id"] = clips.archivo_audio.map(lambda r: r.stem)
clips["origen"] = clips.archivo_audio.map(lambda r: ORIGEN[r.parent.name])
clips["idioma"] = [ATRIBUCION.idioma.get(r.name, "en" if r.parent.name == "en_fleurs" else "es") for r in rutas]
clips.groupby(["origen", "idioma"], sort=False).size().rename("clips").to_frame()

In [ ]:
asr = Transcriptor("faster-whisper")
MODELO_ASR = f"faster-whisper {version('faster-whisper')} · {asr.info['hf']} (int8, CPU, beam 5, vad_filter=True)"
print(asr, f"· cargado en {asr.segundos_carga:.0f} s")
MODELO_ASR

In [ ]:
SEGMENTOS = {}


def transcribir(fila):
    x = cargar_audio(fila.archivo_audio)
    segmentos = asr.segmentos(x, idioma=None if fila.idioma == "qwh" else fila.idioma)
    if fila.id in MULTIHABLANTE:
        segmentos = diarizar(x, segmentos, n_hablantes=MULTIHABLANTE[fila.id])
    SEGMENTOS[fila.id] = segmentos
    return len(x) / SR, sum(b - a for a, b in tramos_de_voz(x))


t0 = time.perf_counter()
clips["duracion_seg"], clips["voz_seg"] = zip(*[transcribir(f) for f in clips.itertuples()])
clips["texto_asr"] = clips.id.map(lambda i: " ".join(s.texto for s in SEGMENTOS[i]))
clips["n_segmentos"] = clips.id.map(lambda i: len(SEGMENTOS[i]))
clips["n_hablantes"] = clips.id.map(lambda i: len({s.hablante for s in SEGMENTOS[i]}) if i in MULTIHABLANTE else 1)


def bucle(i):
    textos = [s.texto.lower().strip(" .¿?") for s in SEGMENTOS[i]]
    mejor, desde, racha = 0, np.nan, 0
    for j, t in enumerate(textos):
        racha = racha + 1 if j and t == textos[j - 1] else 1
        if racha > mejor:
            mejor, desde = racha, SEGMENTOS[i][j - racha + 1].inicio / 60
    return mejor, desde


clips["bucle"], clips["bucle_desde_min"] = zip(*clips.id.map(bucle))
print(f"{len(clips)} clips · {clips.duracion_seg.sum() / 60:.1f} min de audio transcritos en {(time.perf_counter() - t0) / 60:.1f} min")
clips.groupby("origen", sort=False).agg(clips=("id", "size"), minutos=("duracion_seg", lambda s: s.sum() / 60),
                                        voz_pct=("voz_seg", "sum"), segmentos=("n_segmentos", "sum"),
                                        con_bucle=("bucle", lambda s: (s >= 3).sum())
                                        ).assign(voz_pct=lambda d: 100 * d.voz_pct / (60 * d.minutos)).round(1)

**El VAD no lo arregla todo.** Cuando el audio se vuelve débil o ruidoso, o está en un idioma que el modelo no conoce, Whisper se
condiciona en lo que acaba de escribir y repite el mismo segmento varias veces seguidas. `bucle` es la racha más larga de segmentos
idénticos consecutivos de cada clip. Desde 3 se marca como bucle; una racha de 2 puede ser un eco legítimo (en `agustino_2` la
entrevistada repite "Al Mundial" tras el entrevistador). Los clips con bucle no se descartan, porque el resto del texto sirve,
pero son los primeros que hay que revisar a mano:

In [ ]:
clips.loc[clips.bucle >= 3, ["id", "idioma", "duracion_seg", "n_segmentos", "bucle", "bucle_desde_min"]].round(1)

**¿Acierta la diarización?** `entrevista_simulada.wav` es un montaje de 16 turnos con la verdad en `entrevista_simulada.csv`.
Los tiempos de Whisper y los del montaje no coinciden al décimo de segundo, así que cada segmento se empareja con el turno
verdadero de texto más parecido (rapidfuzz) y se compara el hablante.

In [ ]:
from rapidfuzz import fuzz, process

verdad = pd.read_csv(AUDIOS / "es_peru" / "entrevista_simulada.csv")
simulada = a_tabla(SEGMENTOS["entrevista_simulada"])
simulada["verdad"] = [verdad.hablante[process.extractOne(t, verdad.texto, scorer=fuzz.ratio)[2]] for t in simulada.texto]
equivalencia = simulada.groupby("hablante").verdad.agg(lambda s: s.mode()[0])
acierto = (simulada.hablante.map(equivalencia) == simulada.verdad).mean()
print(f"{equivalencia.to_dict()} · {acierto:.0%} de los {len(simulada)} segmentos con el hablante correcto")

clips.loc[clips.id.isin(MULTIHABLANTE), ["id", "duracion_seg", "n_segmentos", "n_hablantes"]].round(1)

**Exportación.** Por clip, un `.srt` (subtítulos con hablante, para VLC, ELAN, Atlas.ti o NVivo) y un `.txt` (turnos legibles),
más un `segmentos.csv` con todos los segmentos. Esta primera exportación es **cruda**: todavía tiene los nombres reales de la
reunión de trabajo y de las entrevistas, así que va a `datos/respaldos/asr_crudo/`, que Git ignora. La versión seudonimizada,
que sí se versiona, se escribe en `datos/asr/` al final de la sección 4.

In [ ]:
def exportar(segmentos, carpeta):
    carpeta.mkdir(parents=True, exist_ok=True)
    for i, segs in segmentos.items():
        (carpeta / f"{i}.srt").write_text(a_srt(segs), encoding="utf-8")
        (carpeta / f"{i}.txt").write_text(a_texto(segs), encoding="utf-8")
    tabla = pd.concat([a_tabla(s).assign(id=i) for i, s in segmentos.items()], ignore_index=True)
    tabla[["id", "inicio", "fin", "hablante", "texto"]].to_csv(carpeta / "segmentos.csv", index=False)
    return tabla


crudo = exportar(SEGMENTOS, DATOS / "respaldos" / "asr_crudo")
print(f"{len(SEGMENTOS)} .srt + {len(SEGMENTOS)} .txt + segmentos.csv ({len(crudo)} segmentos) en datos/respaldos/asr_crudo/")

## 4. Seudonimización

`voz.seudonimizar(texto, reemplazos)` cambia cada nombre de una tabla de equivalencias por su seudónimo con comparación difusa
(rapidfuzz, parecido ≥ 80): atrapa también los nombres que el OCR o Whisper escribieron mal. No detecta nombres por sí sola;
la tabla sale de leer las transcripciones y los textos OCR uno por uno, con especial cuidado en `aud_01_reunion_laboral`, que es
una reunión de trabajo real.

**Criterio.** Un seudónimo por entidad, del tipo `[PERSONA_1]`; las variantes con que el OCR o Whisper escribieron un mismo
nombre apuntan al mismo seudónimo.

- **Se reemplazan** las personas privadas: las vendedoras y el entrevistador de las entrevistas de El Agustino, el colega que
  se menciona de paso en la historia oral, los cuatro firmantes del diploma y el número de registro del diploma.
- **En la reunión de trabajo se reemplaza a toda persona nombrada**, aunque sea pública: decir sobre quién trabaja el equipo ya
  es información de la reunión. Los participantes no se nombran entre sí en la grabación.
- **Se conservan** las personas que ya figuran en el título del archivo y en la atribución que exige la licencia CC BY-SA
  (Aurora Coronado, Don Glines, Elena y Maricela Figueroa): reemplazarlas en el texto no protegería nada. También los personajes
  públicos (futbolistas, Nicolás Maduro en el afiche de la DEA, Diego Rivera en una frase leída de OpenSLR), las marcas de los
  productos fotografiados y las instituciones.

**La tabla no se publica.** Vive en `datos/respaldos/seudonimos.json`, que Git ignora: publicarla desharía la seudonimización.
Tampoco se imprime aquí ningún nombre original; solo seudónimos y conteos.

Además del barrido por nombres se buscan identificadores con forma fija (DNI, celular, correo) con expresiones regulares.

In [ ]:
import json

SEUDONIMOS = json.loads((DATOS / "respaldos" / "seudonimos.json").read_text(encoding="utf-8"))
PATRONES = {"DNI": r"(?<=DNI)\W{0,3}\d{8}\b", "CELULAR": r"\b9\d{2}[ -]?\d{3}[ -]?\d{3}\b",
            "CORREO": r"[\w.+-]+@[\w-]+\.[\w.]+"}
ETIQUETA = re.compile(r"\[([A-Z]+)_?\d*\]")


def anonimizar(texto):
    for tabla in SEUDONIMOS.values():
        texto = seudonimizar(texto, tabla)
    for tipo, patron in PATRONES.items():
        texto = re.sub(patron, f" [{tipo}]", texto)
    return texto


pd.DataFrame({"entidades": {t: len(set(v.values())) for t, v in SEUDONIMOS.items()},
              "claves con variantes": {t: len(v) for t, v in SEUDONIMOS.items()}}).rename_axis("tipo")

In [ ]:
for segmentos in SEGMENTOS.values():
    for s in segmentos:
        s.texto = anonimizar(s.texto)
clips["texto_asr"] = clips.id.map(lambda i: " ".join(s.texto for s in SEGMENTOS[i]))
docs["texto_ocr"] = docs.texto_ocr.map(anonimizar)

textos = pd.concat([docs[["id", "texto_ocr"]].set_axis(["id", "texto"], axis=1).assign(modalidad="documento"),
                    clips[["id", "texto_asr"]].set_axis(["id", "texto"], axis=1).assign(modalidad="audio")])
hallazgos = textos.assign(etiqueta=textos.texto.str.findall(ETIQUETA.pattern)).explode("etiqueta").dropna(subset="etiqueta")
seudonimos = textos.texto.str.findall(r"\[[A-Z]+_?\d*\]").explode().dropna()
resumen = pd.DataFrame({"reemplazos": hallazgos.etiqueta.value_counts(),
                        "entidades distintas": seudonimos.groupby(seudonimos.str.extract(ETIQUETA.pattern, expand=False)).nunique()})
print(f"{len(hallazgos)} reemplazos en {hallazgos.id.nunique()} textos")
resumen.rename_axis("tipo").sort_values("reemplazos", ascending=False)

In [ ]:
hallazgos.pivot_table(index=["modalidad", "id"], columns="etiqueta", aggfunc="size", fill_value=0)

**Revisión de la reunión de trabajo.** Se leyó la transcripción completa de `aud_01_reunion_laboral`. Es una conversación técnica
entre dos personas sobre el monitoreo de un servicio de *scraping* de noticias; ninguna de las dos dice su nombre ni el de la otra.
Abajo, cuánto habla cada una y los segmentos donde se aplicó un seudónimo, para comprobar el reemplazo en contexto:

In [ ]:
reunion = a_tabla(SEGMENTOS["aud_01_reunion_laboral"]).assign(segundos=lambda d: d.fin - d.inicio)
display(reunion.groupby("hablante").agg(segmentos=("texto", "size"), segundos=("segundos", "sum")).round(0))
con_seudonimo = reunion[reunion.texto.str.contains(ETIQUETA.pattern)]
con_seudonimo.assign(texto=con_seudonimo.texto.str.replace(r".*?(.{0,140}\[[A-Z]+_\d+\].{0,80}).*", r"… \1 …", regex=True))[
    ["inicio", "hablante", "texto"]]

**Exportación seudonimizada.** Mismos formatos que en la sección 3, ahora en `datos/asr/` (esta carpeta sí va a Git).

In [ ]:
asr_tabla = exportar(SEGMENTOS, DATOS / "asr")
publicable = pd.concat([asr_tabla.texto, docs.texto_ocr])
quedan = [n for t in SEUDONIMOS.values() for n in t
          if publicable.str.contains(rf"\b{re.escape(n)}\b", case=False, regex=True).any()]
assert not quedan, f"{len(quedan)} nombres de la tabla siguen escritos tal cual"
print(f"{len(SEGMENTOS)} .srt + {len(SEGMENTOS)} .txt + segmentos.csv en datos/asr/ · ningún nombre de la tabla quedó escrito tal cual")

## 5. Duplicados y descartes

**Duplicados.** Huella SHA-256 de cada archivo: dos archivos con la misma huella son byte a byte iguales. Para el PDF se usa
la huella de cada PNG rasterizado. El hash no detecta casi-duplicados (la misma carta fotografiada dos veces), solo copias exactas.

**Ilegibles.** Un documento se descarta si el OCR no encontró texto, si recuperó menos de `MIN_CARACTERES` caracteres
(no alcanza ni para una frase) o si su confianza mediana es menor a `UMBRAL_CONFIANZA`.

**Sin voz.** Un clip se descarta si el VAD encuentra menos de `MIN_VOZ_SEG` segundos de voz o si Whisper no devolvió texto.

In [ ]:
UMBRAL_CONFIANZA, MIN_CARACTERES, MIN_VOZ_SEG = 0.70, 20, 1.0

huella = lambda ruta: hashlib.sha256(Path(ruta).read_bytes()).hexdigest()
docs["sha256"] = docs.archivo.map(huella)
clips["sha256"] = clips.archivo_audio.map(huella)

motivos_doc = pd.Series("", index=docs.index)
motivos_doc[docs.sha256.duplicated()] = "duplicado (mismo hash)"
motivos_doc[(motivos_doc == "") & (docs.lineas == 0)] = "ilegible: el OCR no encontró texto"
motivos_doc[(motivos_doc == "") & (docs.texto_ocr.str.len() < MIN_CARACTERES)] = f"ilegible: menos de {MIN_CARACTERES} caracteres"
motivos_doc[(motivos_doc == "") & (docs.confianza < UMBRAL_CONFIANZA)] = f"ilegible: confianza mediana < {UMBRAL_CONFIANZA}"

motivos_aud = pd.Series("", index=clips.index)
motivos_aud[clips.sha256.duplicated()] = "duplicado (mismo hash)"
motivos_aud[(motivos_aud == "") & (clips.voz_seg < MIN_VOZ_SEG)] = f"sin voz: VAD < {MIN_VOZ_SEG} s"
motivos_aud[(motivos_aud == "") & (clips.texto_asr.str.strip() == "")] = "sin voz: Whisper no devolvió texto"

descartes = pd.concat([
    docs.assign(modalidad="documento", motivo=motivos_doc,
                detalle=docs.texto_ocr.str.len().astype(str) + " car. · confianza " + docs.confianza.round(3).astype(str)),
    clips.assign(modalidad="audio", motivo=motivos_aud, detalle=clips.voz_seg.round(1).astype(str) + " s de voz"),
])[["modalidad", "id", "motivo", "detalle"]]
descartes = descartes[descartes.motivo != ""]

conteo = pd.DataFrame({"revisados": [len(docs), len(clips)],
                       "duplicados": [docs.sha256.duplicated().sum(), clips.sha256.duplicated().sum()]},
                      index=["documento", "audio"])
conteo["ilegibles / sin voz"] = descartes[~descartes.motivo.str.startswith("duplicado")].modalidad.value_counts()
conteo = conteo.fillna(0).astype(int)
conteo["conservados"] = conteo.revisados - conteo.duplicados - conteo["ilegibles / sin voz"]
conteo

In [ ]:
descartes if len(descartes) else print("Ningún archivo descartado.")

In [ ]:
docs_ok = docs[motivos_doc == ""].copy()
clips_ok = clips[motivos_aud == ""].copy()
print(f"Quedan {len(docs_ok)} documentos y {len(clips_ok)} clips ({clips_ok.duracion_seg.sum() / 60:.1f} min)")
docs_ok.tipo.value_counts().rename("documentos").to_frame().T

## 6. Tres miradas al corpus extraído

Los títulos dicen la conclusión; el subtítulo dice qué se grafica. Las duraciones y las longitudes de texto abarcan tres
órdenes de magnitud, así que los histogramas usan tramos que crecen ×2–×2,5 (1-2-5) o ×3 (1-3-10): con tramos de ancho fijo,
casi todo caería en la primera barra. Cada figura se guarda con su versión interactiva y una imagen PNG, para que también
se vea en GitHub.

In [ ]:
import plotly.graph_objects as go

FONDO, TINTA, TINTA_2, REJILLA, EJE = "#fcfcfb", "#0b0b0b", "#52514e", "#e1e0d9", "#c3c2b7"
AZUL, NARANJA, AQUA, AMARILLO = "#2a78d6", "#eb6834", "#1baf7a", "#eda100"

pio.templates["tarea"] = go.layout.Template(layout=dict(
    font=dict(family="Segoe UI, Inter, Arial, sans-serif", size=13, color=TINTA),
    paper_bgcolor=FONDO, plot_bgcolor=FONDO, colorway=[AZUL, NARANJA, AQUA, AMARILLO],
    title=dict(x=0.01, xanchor="left", font=dict(size=17), subtitle=dict(font=dict(size=13, color=TINTA_2))),
    xaxis=dict(gridcolor=REJILLA, linecolor=EJE, zeroline=False, title_font_color=TINTA_2),
    yaxis=dict(gridcolor=REJILLA, linecolor=EJE, zeroline=False, title_font_color=TINTA_2),
    legend=dict(title_font_color=TINTA_2, orientation="h", y=1.0, yanchor="bottom", x=1, xanchor="right"),
    margin=dict(t=110, l=70, r=30, b=60), bargap=0.08,
))
pio.templates.default = "plotly_white+tarea"
pio.renderers.default = "plotly_mimetype+png"
pio.renderers["png"].width, pio.renderers["png"].height, pio.renderers["png"].scale = 920, 500, 1.4

In [ ]:
COLOR_TIPO = {"impreso": AZUL, "foto": NARANJA, "manuscrito": AQUA}
mediana_tipo = docs_ok.groupby("tipo").confianza.median().sort_values()
peor, mejor = mediana_tipo.index[0], mediana_tipo.index[-1]
plural = {"impreso": "impresos", "foto": "fotos", "manuscrito": "manuscritos"}
etiqueta = {t: f"{plural[t]} (n={n})" for t, n in docs_ok.tipo.value_counts().items()}

fig = px.box(docs_ok.assign(grupo=docs_ok.tipo.map(etiqueta)), x="confianza", y="grupo", color="tipo",
             color_discrete_map=COLOR_TIPO, points="all", hover_name="id",
             hover_data={"grupo": False, "tipo": False, "confianza": ":.3f", "lineas": True},
             category_orders={"grupo": [etiqueta[t] for t in mediana_tipo.index[::-1]]},
             labels={"confianza": "confianza mediana del reconocedor por documento", "grupo": ""})
fig.update_traces(marker=dict(size=9, opacity=0.85, line=dict(width=1.5, color=FONDO)), line_width=1.5,
                  jitter=0.45, pointpos=0, boxmean=False, showlegend=False)
fig.update_layout(boxmode="overlay", title=dict(
    text=f"Los {plural[peor]} concentran la confianza más baja del corpus",
    subtitle=dict(text=f"Confianza mediana por tipo: {plural[peor]} {mediana_tipo[peor]:.3f} · "
                       + " · ".join(f"{plural[t]} {mediana_tipo[t]:.3f}" for t in mediana_tipo.index[1:])
                       + f". Cada punto es un documento ({len(docs_ok)} en total).")))
fig.show()

**Confianza no es exactitud.** El reconocedor puede estar muy seguro de un texto equivocado: en `man_03_pizarra_menu_1` lee
"CALDOPGALLINA" con 0,98. La exactitud (CER) se medirá cuando haya `texto_referencia`. Además, los dos manuscritos con más
confianza no son letra a mano: en `man_02_no_a_la_explotacion` el OCR lee sobre todo la pantalla de Stata que está detrás del
cartel, y `man_04_pizarra_menu_2.jpg` contiene la foto del letrero de una tienda de perfumes, no una pizarra. Sin esos dos, la
distancia entre manuscritos y el resto sería todavía mayor. La clasificación de `man_04` queda pendiente de revisión.

In [ ]:
BORDES_DUR = [1, 2, 5, 10, 20, 50, 100, 200, 500, 1000]
TRAMOS_DUR = [f"{a}–{b} s" for a, b in zip(BORDES_DUR[:-1], BORDES_DUR[1:])]
clips_ok["tramo"] = pd.cut(clips_ok.duracion_seg, BORDES_DUR, labels=TRAMOS_DUR, right=False)

largos = clips_ok[clips_ok.duracion_seg >= 60]
peso_largos = largos.duracion_seg.sum() / clips_ok.duracion_seg.sum()
cortos = clips_ok[clips_ok.duracion_seg < 60]
assert set(cortos.origen) <= {"OpenSLR 73", "FLEURS"}

COLOR_ORIGEN = {"OpenSLR 73": AZUL, "FLEURS": NARANJA, "Wikimedia Commons": AQUA, "grabación propia": AMARILLO}
fig = px.histogram(clips_ok, x="tramo", color="origen", color_discrete_map=COLOR_ORIGEN,
                   category_orders={"tramo": TRAMOS_DUR, "origen": list(COLOR_ORIGEN)},
                   labels={"tramo": "duración del clip (tramos 1-2-5)", "origen": ""})
fig.update_layout(yaxis_title="clips", title=dict(
    text=f"{len(largos)} grabaciones largas concentran el {peso_largos:.0%} del audio; los otros {len(cortos)} clips son frases cortas",
    subtitle=dict(text=f"Duración de los {len(clips_ok)} clips ({clips_ok.duracion_seg.sum() / 60:.1f} min). "
                       f"Mediana: {clips_ok.duracion_seg.median():.1f} s; el más largo dura {clips_ok.duracion_seg.max() / 60:.1f} min.")))
fig.show()

In [ ]:
BORDES_LEN = [0, 10, 30, 100, 300, 1000, 3000, 10000, 30000]
TRAMOS_LEN = ["<10", "10–30", "30–100", "100–300", "300–1k", "1k–3k", "3k–10k", "10k–30k"]

textos = pd.concat([
    docs_ok.assign(modalidad="documento (OCR)", texto=docs_ok.texto_ocr)[["id", "modalidad", "tipo", "texto"]],
    clips_ok.assign(modalidad="audio (ASR)", tipo=clips_ok.origen, texto=clips_ok.texto_asr)[["id", "modalidad", "tipo", "texto"]],
], ignore_index=True)
textos["caracteres"] = textos.texto.str.len()
textos["tramo"] = pd.cut(textos.caracteres, BORDES_LEN, labels=TRAMOS_LEN, right=False)

conteo = textos.tramo.value_counts().reindex(TRAMOS_LEN)
primero, ultimo = conteo[conteo > 0].index[[0, -1]]
en_primero, en_ultimo = textos[textos.tramo == primero], textos[textos.tramo == ultimo]
del_pdf = en_ultimo.id.str.startswith(PDF.stem)
grabacion_larga = en_ultimo.id.isin(largos.id)
assert (del_pdf | grabacion_larga).all() and (en_primero.modalidad == "audio (ASR)").mean() > 0.5

mediana_mod = textos.groupby("modalidad").caracteres.median()
fig = px.histogram(textos, x="tramo", color="modalidad", category_orders={"tramo": TRAMOS_LEN},
                   color_discrete_map={"documento (OCR)": AZUL, "audio (ASR)": NARANJA},
                   labels={"tramo": "longitud del texto en caracteres (tramos 1-3-10)", "modalidad": ""})
fig.update_layout(yaxis_title="textos", barmode="stack", margin_t=130, title=dict(
    text="Las frases de audio forman el extremo corto; el PDF y las grabaciones largas, el largo",
    subtitle=dict(text=f"Tramo {primero}: {(en_primero.modalidad == 'audio (ASR)').sum()} de {len(en_primero)} textos son clips. "
                       f"Tramo {ultimo}: {del_pdf.sum()} páginas del PDF y {grabacion_larga.sum()} grabaciones largas.<br>Mediana: "
                       + " · ".join(f"{m} {v:,.0f} car." for m, v in mediana_mod.items()))))
fig.show()

## 7. Los extremos, contra los histogramas

Los tres textos más largos y más cortos, y los tres clips más largos y más cortos, con el tramo del histograma en que cae cada uno.

**Cuadre.** Si los histogramas están bien, leerlos desde la derecha tiene que dar los tramos de los tres más largos, y leerlos
desde la izquierda, los de los tres más cortos. Por ejemplo, si la última barra tiene altura 1 y la penúltima 4, el más largo
está en la última y el segundo y el tercero en la penúltima. `cuadre()` reconstruye esa lectura a partir de los conteos por
tramo (los mismos que dibujó Plotly) y la compara con los extremos reales; un `assert` detiene el notebook si no coinciden.

In [ ]:
def extremos(tabla, columna, n=3):
    return pd.concat([tabla.nlargest(n, columna).assign(extremo="más largo"),
                      tabla.nsmallest(n, columna).assign(extremo="más corto")])


def cuadre(tabla, columna, n=3):
    conteo = tabla.tramo.value_counts().reindex(tabla.tramo.cat.categories)
    leido = np.repeat(conteo.index.astype(str), conteo.values)
    esperado = {"más largo": list(leido[::-1][:n]), "más corto": list(leido[:n])}
    real = {"más largo": tabla.nlargest(n, columna).tramo.astype(str).tolist(),
            "más corto": tabla.nsmallest(n, columna).tramo.astype(str).tolist()}
    assert esperado == real, (esperado, real)
    return pd.DataFrame({"según el histograma": esperado, "según los datos": real}).map(" · ".join)


ext_textos = extremos(textos, "caracteres")
ext_textos["inicio del texto"] = ext_textos.texto.str.slice(0, 100)
ext_textos[["extremo", "id", "modalidad", "caracteres", "tramo", "inicio del texto"]]

In [ ]:
cuadre(textos, "caracteres")

In [ ]:
ext_clips = extremos(clips_ok, "duracion_seg")
ext_clips[["extremo", "id", "origen", "duracion_seg", "tramo", "n_hablantes"]].round(1)

In [ ]:
cuadre(clips_ok, "duracion_seg")

## 8. Exportación final

**`texto_referencia` donde sale gratis:**

- los 40 clips de OpenSLR 73 traen su transcripción oficial en `audios/es_peru/transcripciones.csv`;
- las 8 páginas del PDF del MINJUS traen capa de texto, que se extrae con `pypdf`. Ojo: es el texto que el PDF tiene incrustado,
  no una transcripción hecha a mano; el orden de encabezados y notas al pie puede diferir del orden de lectura del OCR.

El resto queda vacío para completarlo a mano. La referencia pasa por la misma seudonimización que el texto extraído, para que
la comparación posterior (CER/WER) sea justa.

In [ ]:
from pypdf import PdfReader

referencia_pdf = {f"{PDF.stem}_p{i:02d}": " ".join((p.extract_text() or "").split())
                  for i, p in enumerate(PdfReader(PDF).pages, start=1)}
referencia_openslr = pd.read_csv(AUDIOS / "es_peru" / "transcripciones.csv").assign(
    id=lambda d: d.archivo.str.removesuffix(".wav")).set_index("id").texto.to_dict()
print(f"Referencias: {len(referencia_pdf)} páginas del PDF, {len(referencia_openslr)} clips de OpenSLR")

In [ ]:
FUENTE = fuentes.set_index("id").fuente_url

documentos = pd.DataFrame({
    "id": docs_ok.id,
    "archivo": docs_ok.archivo.map(lambda r: r.as_posix()),
    "tipo": docs_ok.tipo,
    "fuente": docs_ok.fuente_id.map(FUENTE),
    "texto_ocr": docs_ok.texto_ocr,
    "modelo_ocr": MODELO_OCR,
    "texto_referencia": docs_ok.id.map(referencia_pdf).fillna("").map(anonimizar),
}).reset_index(drop=True)

transcripciones = pd.DataFrame({
    "id": clips_ok.id,
    "archivo_audio": clips_ok.archivo_audio.map(lambda r: r.as_posix()),
    "duracion_seg": clips_ok.duracion_seg.round(2),
    "idioma": clips_ok.idioma,
    "texto_asr": clips_ok.texto_asr,
    "modelo_asr": MODELO_ASR,
    "n_hablantes": clips_ok.n_hablantes,
    "texto_referencia": clips_ok.id.map(referencia_openslr).fillna("").map(anonimizar),
}).reset_index(drop=True)

assert documentos.fuente.notna().all() and documentos.id.is_unique and transcripciones.id.is_unique
documentos.to_csv(DATOS / "documentos.csv", index=False)
transcripciones.to_csv(DATOS / "transcripciones.csv", index=False)
for nombre, df in [("documentos", documentos), ("transcripciones", transcripciones)]:
    con_ref = (df.texto_referencia != "").sum()
    print(f"datos/{nombre}.csv: {df.shape} · {con_ref} filas con texto_referencia")

In [ ]:
print(documentos.shape)
documentos.head()

In [ ]:
print(transcripciones.shape)
transcripciones.head()

**Pendiente para la Parte 2:** completar a mano `texto_referencia` en una muestra de fotos, manuscritos y entrevistas para medir
CER y WER fuera de los casos fáciles (texto impreso digital y frases leídas).